# Avaliação comparativa de modelos — RNF04Compara modelos de linguagem sobre o **mesmo** conjunto de avaliação, medindo o pipelinede estruturação real do `ai-structuring-service` — prompt versionado, barreira de schema epolítica de retentativa incluídas.Implementa a especificação de [`docs/04-ia/metricas-qualidade.md`](../docs/04-ia/metricas-qualidade.md).Entregável da 3ª parcial (20/10), cards #215 a #218.## O que este notebook prova, e o que não prova**Prova, com número real:** conformidade de schema, taxa de falha, latência p50/p95, tokens pordemanda, e — quando o conjunto tiver casos anotados — cobertura, taxa de alucinação ereconhecimento de ausência.**Não prova:** acurácia dos campos de texto livre. O critério de comparação fixado pelo próprioprojeto é *equivalência semântica julgada por dois humanos* para `tema`, `nicho` e `publico_alvo`,e escala de 1 a 5 para `ementa` e `objetivos_aprendizagem`. Comparar esses campos por igualdade destring produziria acurácia artificialmente baixa, então eles ficam **fora** do número automático esaem listados como pendentes de julgamento. Dos oito campos, **três** são pontuáveis por máquina:`numero_participantes`, `carga_horaria` e `formato`.**Também não prova nada sobre o cliente real enquanto o conjunto for sintético.** A célula decarregamento diz em quantos casos reais anonimizados o conjunto está. Número medido sobre casosintetizado serve para validar o arnês, não para reportar qualidade.

## 1. AmbienteO notebook roda a partir da raiz do repositório e usa o pacote do serviço direto da árvore, seminstalar nada além das dependências do próprio `ai-structuring-service`.Variáveis obrigatórias (as mesmas do serviço, ver `services/ai-structuring-service/.env.example`):`ENVIRONMENT`, `LOG_LEVEL`, `AI_STRUCTURING_PORT`, `PIPELINE_SERVICE_URL`, `LLM_PROVIDER`,`LLM_MODEL`, `LLM_API_KEY`, `LLM_TIMEOUT_SECONDS`, `LLM_MAX_RETRIES`, `LLM_TEMPERATURE`.Nenhuma chave entra aqui no código (RNF11). Para rodar contra o OpenRouter, exporte`OPENROUTER_API_KEY` no ambiente antes de abrir o notebook.

In [1]:
import asyncio
import os
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SERVICO = RAIZ / "services" / "ai-structuring-service"
sys.path.insert(0, str(SERVICO))

# Configuracao minima para o notebook rodar com o provedor mock, sem chave.
# Quem roda contra o OpenRouter sobrescreve LLM_PROVIDER e LLM_BASE_URL na secao 3.
os.environ.setdefault("ENVIRONMENT", "development")
os.environ.setdefault("LOG_LEVEL", "WARNING")
os.environ.setdefault("AI_STRUCTURING_PORT", "8003")
os.environ.setdefault("PIPELINE_SERVICE_URL", "http://localhost:8001")
os.environ.setdefault("LLM_PROVIDER", "mock")
os.environ.setdefault("LLM_MODEL", "mock-1")
os.environ.setdefault("LLM_API_KEY", "")
os.environ.setdefault("LLM_TIMEOUT_SECONDS", "30")
os.environ.setdefault("LLM_MAX_RETRIES", "2")
os.environ.setdefault("LLM_TEMPERATURE", "0")

print("raiz do repositorio:", RAIZ)

raiz do repositorio: D:\Projetos\pcc-157


## 2. Conjunto de avaliaçãoCarrega `tests/fixtures/` no layout que o `README.md` de lá especifica: entrada em`emails/`, `transcricoes/` ou `mensagens/`, gabarito de mesmo nome em `gabaritos/`.**Leia o resumo com atenção.** Ele diz quantos casos são reais anonimizados e quantos sãosintéticos. A meta do projeto é 20 casos reais anotados; enquanto o número de reais for zero,nenhuma métrica de qualidade daqui é reportável como medição do produto.

In [2]:
from app.evaluation.dataset import Procedencia, carregar_conjunto

conjunto = carregar_conjunto()
casos_por_id = {c.identificador: c for c in conjunto}

reais = conjunto.por_procedencia(Procedencia.REAL)
print(conjunto.resumo)

if not reais:
    print()
    print("ATENCAO: nenhum caso real anonimizado no conjunto.")
    print("As metricas abaixo validam o arnes, nao a qualidade do produto.")
    print("Para a 3a parcial, o conjunto precisa dos casos reais anotados por humano.")

if conjunto.sem_gabarito:
    print()
    print(f"{len(conjunto.sem_gabarito)} entradas sem gabarito (ficam fora da medicao):")
    for caminho in conjunto.sem_gabarito:
        print("  -", caminho)

print()
for caso in conjunto:
    print(
        f"{caso.identificador}  canal={caso.canal:13s} proc={caso.procedencia.value:18s} "
        f"com base={len(caso.com_base)} sem base={len(caso.sem_base)}"
    )

3 casos anotados (0 reais anonimizados, 3 sinteticos); 0 entradas sem gabarito

ATENCAO: nenhum caso real anonimizado no conjunto.
As metricas abaixo validam o arnes, nao a qualidade do produto.
Para a 3a parcial, o conjunto precisa dos casos reais anotados por humano.

sint-001  canal=emails        proc=sintetico          com base=8 sem base=0
sint-002  canal=mensagens     proc=sintetico          com base=5 sem base=3
sint-003  canal=transcricoes  proc=sintetico          com base=7 sem base=1


## 3. Catálogo de modelos e provedorO catálogo vive em `app/evaluation/modelos.toml`, versionado — comparação só é defensável seoutra pessoa puder repeti-la com a mesma lista.O OpenRouter entra pelo `HttpLLMProvider`, que já é um cliente de `/chat/completions` compatívelcom OpenAI. **Não há provedor novo a escrever:** basta apontar `LLM_BASE_URL` para o OpenRouter etrocar o modelo por chamada.Para rodar de verdade, defina `OPENROUTER_API_KEY` no ambiente. Sem ela, o notebook cai noprovedor mock e avisa — assim ele roda ponta a ponta em qualquer máquina, inclusive no CI.

In [3]:
from app.core.config import Settings, get_settings
from app.evaluation.catalog import carregar_catalogo, selecionar
from app.providers.base import LLMProvider
from app.providers.http_llm_provider import HttpLLMProvider
from app.providers.mock_provider import MockLLMProvider

CHAVE = os.environ.get("OPENROUTER_API_KEY", "").strip()
USAR_OPENROUTER = bool(CHAVE)

catalogo = carregar_catalogo()

# Subconjunto: rodar os quatro gasta credito. Comece por um, confira, depois amplie.
SLUGS = None  # ex.: ["openai/gpt-4o-mini"]
modelos = selecionar(catalogo, SLUGS)

print(f"{len(modelos)} modelos no recorte:")
for m in modelos:
    pendente = " [preco pendente]" if m.precos_pendentes else ""
    print(f"  {m.apelido:20s} {m.slug:40s} ctx={m.janela_de_contexto:>9,}{pendente}")

if any(m.precos_pendentes for m in modelos):
    print()
    print("Preco nao consultado em ao menos um modelo: a coluna de custo sai como n/d.")
    print("Preencher usd_por_mtok_* e preco_consultado_em em app/evaluation/modelos.toml.")


def fabrica_de_provedor(slug: str, settings: Settings) -> LLMProvider:
    """Cria o provedor para um modelo do catalogo."""
    if not USAR_OPENROUTER:
        return MockLLMProvider(model=slug)
    config = settings.model_copy(
        update={
            "llm_provider": "http",
            "llm_base_url": "https://openrouter.ai/api/v1",
            "llm_completions_path": "/chat/completions",
            "llm_model": slug,
            "llm_api_key": CHAVE,
        }
    )
    return HttpLLMProvider(config)


settings = get_settings()
print()
print("provedor:", "OpenRouter" if USAR_OPENROUTER else "mock (sem OPENROUTER_API_KEY)")

4 modelos no recorte:
  Sonnet 4.5           anthropic/claude-sonnet-4.5              ctx=  200,000 [preco pendente]
  GPT-4o mini          openai/gpt-4o-mini                       ctx=  128,000 [preco pendente]
  Gemini 2.0 Flash     google/gemini-2.0-flash-001              ctx=1,000,000 [preco pendente]
  Llama 3.3 70B        meta-llama/llama-3.3-70b-instruct        ctx=  131,072 [preco pendente]

Preco nao consultado em ao menos um modelo: a coluna de custo sai como n/d.
Preencher usd_por_mtok_* e preco_consultado_em em app/evaluation/modelos.toml.

provedor: mock (sem OPENROUTER_API_KEY)


## 4. RodadaExecuta cada caso em cada modelo e **grava a saída bruta antes de calcular qualquer métrica**.Chamada de LLM custa dinheiro e tempo; reanálise não pode exigir rodada nova.`REPETICOES` acima de 1 alimenta a métrica de estabilidade (RNF03), que exige a mesma entrada maisde uma vez. Com temperatura 0 a variação esperada é baixa, mas medir é o ponto.

In [4]:
from datetime import UTC, datetime

from app.evaluation.runner import executar, salvar

REPETICOES = 1

progresso = []


def acompanhar(execucao):
    marca = "ok " if execucao.curso is not None else "FALHA"
    progresso.append(execucao)
    print(f"  [{len(progresso):>3}] {marca} {execucao.modelo:40s} {execucao.caso}")


print(f"rodando {len(modelos)} modelos x {len(conjunto)} casos x {REPETICOES} repeticao(oes)")
rodada = await executar(
    casos=list(conjunto),
    modelos=modelos,
    fabrica_de_provedor=fabrica_de_provedor,
    settings=settings,
    repeticoes=REPETICOES,
    ao_terminar_caso=acompanhar,
)

destino = RAIZ / "docs" / "04-ia" / "avaliacoes" / "dados" / (
    f"{datetime.now(UTC):%Y-%m-%d}-{rodada.prompt_version}.json"
)
salvar(rodada, destino)
print()
print(f"{rodada.total} execucoes | prompt {rodada.prompt_version}")
print("saidas brutas em:", destino.relative_to(RAIZ))

reportados = {e.modelo_reportado for e in rodada.execucoes if e.modelo_reportado}
print("modelos reportados pelo provedor:", sorted(reportados))

rodando 4 modelos x 3 casos x 1 repeticao(oes)
  [  1] ok  anthropic/claude-sonnet-4.5              sint-001
  [  2] ok  anthropic/claude-sonnet-4.5              sint-002
  [  3] ok  anthropic/claude-sonnet-4.5              sint-003
  [  4] ok  openai/gpt-4o-mini                       sint-001
  [  5] ok  openai/gpt-4o-mini                       sint-002
  [  6] ok  openai/gpt-4o-mini                       sint-003
  [  7] ok  google/gemini-2.0-flash-001              sint-001
  [  8] ok  google/gemini-2.0-flash-001              sint-002
  [  9] ok  google/gemini-2.0-flash-001              sint-003
  [ 10] ok  meta-llama/llama-3.3-70b-instruct        sint-001
  [ 11] ok  meta-llama/llama-3.3-70b-instruct        sint-002
  [ 12] ok  meta-llama/llama-3.3-70b-instruct        sint-003

12 execucoes | prompt v5
saidas brutas em: docs\04-ia\avaliacoes\dados\2026-10-28-v5.json
modelos reportados pelo provedor: ['mock-1']


### Divergência entre modelo pedido e modelo servidoO roteamento do OpenRouter pode atender a chamada com um modelo diferente do pedido. A rodadaguarda os dois: o slug do catálogo, que é a chave de agrupamento, e o que o provedor reportou.Divergência aqui é achado, não detalhe — invalida a comparação se passar em silêncio.

In [5]:
divergentes = [
    (e.modelo, e.modelo_reportado, e.caso)
    for e in rodada.execucoes
    if e.modelo_reportado and e.modelo_reportado != e.modelo
]

if not divergentes:
    print("Nenhuma divergencia: todo modelo servido corresponde ao pedido.")
else:
    print(f"{len(divergentes)} execucoes servidas por modelo diferente do pedido:")
    for pedido, servido, caso in divergentes[:20]:
        print(f"  {caso}: pedido {pedido} -> servido {servido}")
    print()
    print("Com provedor mock isso e esperado (o mock reporta o proprio nome).")
    print("Com OpenRouter, investigar antes de usar os numeros.")

12 execucoes servidas por modelo diferente do pedido:
  sint-001: pedido anthropic/claude-sonnet-4.5 -> servido mock-1
  sint-002: pedido anthropic/claude-sonnet-4.5 -> servido mock-1
  sint-003: pedido anthropic/claude-sonnet-4.5 -> servido mock-1
  sint-001: pedido openai/gpt-4o-mini -> servido mock-1
  sint-002: pedido openai/gpt-4o-mini -> servido mock-1
  sint-003: pedido openai/gpt-4o-mini -> servido mock-1
  sint-001: pedido google/gemini-2.0-flash-001 -> servido mock-1
  sint-002: pedido google/gemini-2.0-flash-001 -> servido mock-1
  sint-003: pedido google/gemini-2.0-flash-001 -> servido mock-1
  sint-001: pedido meta-llama/llama-3.3-70b-instruct -> servido mock-1
  sint-002: pedido meta-llama/llama-3.3-70b-instruct -> servido mock-1
  sint-003: pedido meta-llama/llama-3.3-70b-instruct -> servido mock-1

Com provedor mock isso e esperado (o mock reporta o proprio nome).
Com OpenRouter, investigar antes de usar os numeros.


## 5. Métricas por modeloCada métrica vem com a meta do documento ao lado. `n/d` significa **não medido**, nunca zero —conjunto sem campo sem base não tem taxa de alucinação, e imprimir 0% afirmaria algo que não foimedido.Na taxa de alucinação e na taxa de falha, menor é melhor.

In [6]:
from app.evaluation.metrics import METAS, medir

def pct(valor):
    return "   n/d" if valor is None else f"{valor:6.0%}"


def marca(medicao, nome):
    veredito = medicao.atende(nome)
    return " " if veredito is None else ("ok" if veredito else "!!")


medicoes = [medir(m.slug, casos_por_id, rodada.execucoes) for m in modelos]

cabecalho = (
    f"{'modelo':20s} {'casos':>5s} {'acur':>6s}   {'essenc':>6s}   {'cobert':>6s}   "
    f"{'alucin':>6s}   {'ausenc':>6s}   {'schema':>6s}   {'falha':>6s}   "
    f"{'evid':>6s}   {'cit.inv':>7s}  "
)
print(cabecalho)
print("-" * len(cabecalho))
for modelo, d in zip(modelos, medicoes):
    print(
        f"{modelo.apelido:20s} {d.casos:>5d} "
        f"{pct(d.acuracia_global)} {marca(d, 'acuracia_global')} "
        f"{pct(d.acuracia_essenciais)} {marca(d, 'acuracia_essenciais')} "
        f"{pct(d.cobertura)} {marca(d, 'cobertura')} "
        f"{pct(d.taxa_de_alucinacao)} {marca(d, 'taxa_de_alucinacao')} "
        f"{pct(d.reconhecimento_de_ausencia)} {marca(d, 'reconhecimento_de_ausencia')} "
        f"{pct(d.conformidade_de_schema)} {marca(d, 'conformidade_de_schema')} "
        f"{pct(d.taxa_de_falha)} {marca(d, 'taxa_de_falha')} "
        f"{pct(d.cobertura_de_evidencia)} {marca(d, 'cobertura_de_evidencia')} "
        f"{pct(d.taxa_de_citacao_invalida)} {marca(d, 'taxa_de_citacao_invalida')}"
    )

print()
print("metas:", {k: f"{v:.0%}" for k, v in METAS.items()})
print()
print("fora do numero automatico (exigem julgamento humano):")
print("  ", ", ".join(medicoes[0].pendentes_de_julgamento) if medicoes else "-")

modelo               casos   acur   essenc   cobert   alucin   ausenc   schema    falha     evid   cit.inv  
------------------------------------------------------------------------------------------------------------
Sonnet 4.5               3    33% !!    50% !!    85% ok   100% !!     0% !!   100% ok     0% ok   100% ok     0% ok
GPT-4o mini              3    33% !!    50% !!    85% ok   100% !!     0% !!   100% ok     0% ok   100% ok     0% ok
Gemini 2.0 Flash         3    33% !!    50% !!    85% ok   100% !!     0% !!   100% ok     0% ok   100% ok     0% ok
Llama 3.3 70B            3    33% !!    50% !!    85% ok   100% !!     0% !!   100% ok     0% ok   100% ok     0% ok

metas: {'acuracia_essenciais': '80%', 'acuracia_global': '75%', 'cobertura': '85%', 'taxa_de_alucinacao': '5%', 'reconhecimento_de_ausencia': '90%', 'estabilidade': '90%', 'conformidade_de_schema': '95%', 'taxa_de_falha': '5%', 'cobertura_de_evidencia': '95%', 'taxa_de_citacao_invalida': '5%'}

fora do numero au

### Acurácia por campo, operação e custoA acurácia por campo só existe para os três campos de critério automático. Latência e tokens sãoreais em qualquer rodada; o custo depende do preço estar preenchido no catálogo.

In [7]:
for modelo, d in zip(modelos, medicoes):
    print(f"{modelo.apelido}")
    for campo, valor in d.acuracia_por_campo.items():
        print(f"   {campo:24s} {pct(valor)}")
    p50 = "n/d" if d.latencia_p50_ms is None else f"{d.latencia_p50_ms:.0f} ms"
    p95 = "n/d" if d.latencia_p95_ms is None else f"{d.latencia_p95_ms:.0f} ms"
    tokens = "n/d" if d.tokens_por_demanda is None else f"{d.tokens_por_demanda:.0f}"
    print(f"   {'latencia p50 / p95':24s} {p50} / {p95}")
    print(f"   {'tokens por demanda':24s} {tokens}")

    if d.tokens_por_demanda and not modelo.precos_pendentes:
        entrada = sum(e.tokens_entrada or 0 for e in rodada.execucoes if e.modelo == modelo.slug)
        saida = sum(e.tokens_saida or 0 for e in rodada.execucoes if e.modelo == modelo.slug)
        custo = modelo.custo_usd(entrada, saida)
        por_caso = custo / d.casos if d.casos else 0
        print(f"   {'custo da rodada (USD)':24s} {custo:.6f}  ({por_caso:.6f} por demanda)")
    else:
        print(f"   {'custo da rodada (USD)':24s} n/d (preco nao consultado no catalogo)")
    print()

Sonnet 4.5
   numero_participantes         0%
   carga_horaria               50%
   formato                     50%
   latencia p50 / p95       0 ms / 0 ms
   tokens por demanda       5617
   custo da rodada (USD)    n/d (preco nao consultado no catalogo)

GPT-4o mini
   numero_participantes         0%
   carga_horaria               50%
   formato                     50%
   latencia p50 / p95       0 ms / 0 ms
   tokens por demanda       5617
   custo da rodada (USD)    n/d (preco nao consultado no catalogo)

Gemini 2.0 Flash
   numero_participantes         0%
   carga_horaria               50%
   formato                     50%
   latencia p50 / p95       0 ms / 0 ms
   tokens por demanda       5617
   custo da rodada (USD)    n/d (preco nao consultado no catalogo)

Llama 3.3 70B
   numero_participantes         0%
   carga_horaria               50%
   formato                     50%
   latencia p50 / p95       0 ms / 0 ms
   tokens por demanda       5617
   custo da rodada (USD)    n/

### Estabilidade (RNF03)Mesma entrada, execuções repetidas, mesma saída. Exige `REPETICOES` maior que 1 — com umaexecução por caso não há o que comparar e a métrica sai como `n/d`, que é o correto.

In [8]:
from app.evaluation.metrics import estabilidade

for modelo in modelos:
    por_caso = []
    for identificador in casos_por_id:
        repeticoes = [
            e
            for e in rodada.execucoes
            if e.modelo == modelo.slug and e.caso == identificador
        ]
        valor = estabilidade(repeticoes)
        if valor is not None:
            por_caso.append(valor)
    if por_caso:
        media = sum(por_caso) / len(por_caso)
        veredito = "ok" if media >= METAS["estabilidade"] else "!!"
        print(f"{modelo.apelido:20s} {media:6.0%} {veredito}  ({len(por_caso)} casos)")
    else:
        print(f"{modelo.apelido:20s}    n/d  (defina REPETICOES > 1 na secao 4)")

Sonnet 4.5              n/d  (defina REPETICOES > 1 na secao 4)
GPT-4o mini             n/d  (defina REPETICOES > 1 na secao 4)
Gemini 2.0 Flash        n/d  (defina REPETICOES > 1 na secao 4)
Llama 3.3 70B           n/d  (defina REPETICOES > 1 na secao 4)


## 6. Relatório da rodadaGera o arquivo no caminho que `metricas-qualidade.md` especifica —`docs/04-ia/avaliacoes/AAAA-MM-DD-<versao-do-prompt>.md` — para a rodada ficar versionada juntocom o código e comparável com a próxima.

In [9]:
linhas = [
    f"# Avaliação de modelos — {datetime.now(UTC):%d/%m/%Y}",
    "",
    f"- **Versão do prompt:** `{rodada.prompt_version}`",
    f"- **Provedor:** {'OpenRouter' if USAR_OPENROUTER else 'mock (sem chave)'}",
    f"- **Conjunto:** {conjunto.resumo}",
    f"- **Execuções:** {rodada.total} ({len(modelos)} modelos x {len(conjunto)} casos "
    f"x {REPETICOES} repetição)",
    "",
]

if not reais:
    linhas += [
        "> **Os números abaixo não são reportáveis como medição do produto.** O conjunto não tem",
        "> caso real anonimizado; estes valores validam o arnês de avaliação, não a qualidade da",
        "> estruturação. Para a entrega, o conjunto precisa dos casos reais anotados por humano.",
        "",
    ]

linhas += [
    "## Métricas",
    "",
    "| Modelo | Casos | Acurácia | Essenciais | Cobertura | Alucinação | Ausência | Schema | Falha | Evidência | Citação inválida |",
    "|---|--:|--:|--:|--:|--:|--:|--:|--:|--:|--:|",
]
for modelo, d in zip(modelos, medicoes):
    linhas.append(
        f"| {modelo.apelido} | {d.casos} | {pct(d.acuracia_global).strip()} | "
        f"{pct(d.acuracia_essenciais).strip()} | {pct(d.cobertura).strip()} | "
        f"{pct(d.taxa_de_alucinacao).strip()} | {pct(d.reconhecimento_de_ausencia).strip()} | "
        f"{pct(d.conformidade_de_schema).strip()} | {pct(d.taxa_de_falha).strip()} | "
        f"{pct(d.cobertura_de_evidencia).strip()} | {pct(d.taxa_de_citacao_invalida).strip()} |"
    )

linhas += [
    "",
    "## Limitações desta rodada",
    "",
    "- A acurácia automática cobre apenas "
    f"`{'`, `'.join(medicoes[0].acuracia_por_campo)}`. Os campos "
    f"`{'`, `'.join(medicoes[0].pendentes_de_julgamento)}` exigem julgamento humano pelo "
    "critério fixado em `metricas-qualidade.md` e ficam fora do número.",
    "- Custo por demanda sai como n/d enquanto o preço de cada modelo não for consultado e "
    "datado em `app/evaluation/modelos.toml`.",
    "- Evidência tende a 100%: a validação já recusa campo preenchido sem citação e citação "
    "que não corresponde a um trecho real (RNF27), então só a execução aprovada chega aqui. "
    "O número confirma que a barreira funciona; não mede a qualidade da citação em si.",
    "",
    "## Saídas brutas",
    "",
    f"`{destino.relative_to(RAIZ).as_posix()}`",
    "",
]

relatorio = RAIZ / "docs" / "04-ia" / "avaliacoes" / (
    f"{datetime.now(UTC):%Y-%m-%d}-{rodada.prompt_version}.md"
)
relatorio.parent.mkdir(parents=True, exist_ok=True)
relatorio.write_text("\n".join(linhas) + "\n", encoding="utf-8")
print("relatorio escrito em:", relatorio.relative_to(RAIZ))
print()
print("\n".join(linhas[:14]))

relatorio escrito em: docs\04-ia\avaliacoes\2026-10-28-v5.md

# Avaliação de modelos — 28/10/2026

- **Versão do prompt:** `v5`
- **Provedor:** mock (sem chave)
- **Conjunto:** 3 casos anotados (0 reais anonimizados, 3 sinteticos); 0 entradas sem gabarito
- **Execuções:** 12 (4 modelos x 3 casos x 1 repetição)

> **Os números abaixo não são reportáveis como medição do produto.** O conjunto não tem
> caso real anonimizado; estes valores validam o arnês de avaliação, não a qualidade da
> estruturação. Para a entrega, o conjunto precisa dos casos reais anotados por humano.

## Métricas

| Modelo | Casos | Acurácia | Essenciais | Cobertura | Alucinação | Ausência | Schema | Falha | Evidência | Citação inválida |


## 7. O que falta para a 3ª parcial1. **Casos reais anotados.** É o único item que nenhum código resolve: 20 demandas reais do   cliente, anonimizadas e com gabarito anotado por humano, em `tests/fixtures/`. Enquanto o   conjunto for sintético, as métricas de qualidade validam o arnês e não o produto.2. **Preço dos modelos.** Consultar `openrouter.ai/models`, preencher `usd_por_mtok_*` e   atualizar `preco_consultado_em` em `app/evaluation/modelos.toml`. Sem isso não há coluna de   custo, que é metade da decisão de escolha de modelo.3. **Julgamento humano dos campos de texto livre.** Dois avaliadores sobre `tema`, `nicho` e   `publico_alvo`, e a escala de 1 a 5 em `ementa` e `objetivos_aprendizagem`, conforme o   critério já acordado. Sem isso, cinco dos oito campos ficam sem acurácia.4. **Gráfico de comparação.** Fica para a primeira rodada real: com quatro modelos devolvendo os   mesmos números do mock, um gráfico mostraria quatro barras idênticas e enganaria mais do que a   tabela.